[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyneuro/Chapter_Colabs/blob/main/Colab_F.ipynb)

# Set F — Small Networks: STM & WTA (LEGO–Colab)
**Author:** Neural Engineering Laboratory, University of Missouri-
Gregory Glickert, Khuram Choudhry, Ziao Chen, Satish S. Nair

---

# **Variable Key & Roadmap**

### **The "Levers" (Network Parameters)**
These parameters control the biophysical behavior of the circuits you will build below.

| **Variable** | **Definition** | **Unit** | **Instructional Role** |
| :--- | :--- | :--- | :--- |
| **`weight_EE`** | Excitatory to Excitatory strength | μS | Controls "Reverberation" / Persistence duration in STM. |
| **`E2I_weight`** | Excitatory to Inhibitory strength | μS | Controls how easily a Principal cell recruits its partner interneuron. |
| **`II_weight`** | Inhibitory to Excitatory strength | μS | Controls competitive suppression (The "Veto" in Winner-Take-All). |
| **`syn_delay`** | Synaptic delay | ms | Affects the timing and stability of decisions. |
| **`g_pas`** | Passive Leak Conductance | S/cm² | Determines the "forgetfulness" or decay rate of the membrane. |

---

### **The Roadmap**
This module explores how small groups of neurons work together to perform basic cognitive functions.
* **[F0 Starter](https://colab.research.google.com/drive/1OdhhV057bx5SmkKs4LG1UKrTQtSZ4bXc#scrollTo=QwiJS79RIitS):** Initializing the multi-cell "Engine" and helper builders.
* **[F1 Excitatory STM](https://colab.research.google.com/drive/1OdhhV057bx5SmkKs4LG1UKrTQtSZ4bXc#scrollTo=v7ZlK6x8Yp6P):** Investigating if a brief pulse can trigger a memory that outlasts the stimulus.
* **[W — Winner-Take-All](https://colab.research.google.com/drive/1OdhhV057bx5SmkKs4LG1UKrTQtSZ4bXc#scrollTo=874e5088):** Using mutual inhibition to force the network to "choose" a winner.
* **[Model Validation](https://colab.research.google.com/drive/1OdhhV057bx5SmkKs4LG1UKrTQtSZ4bXc#scrollTo=95cbf906):** Verifying the synaptic logic with a simplified 2-cell test.
* **[Practice & Reflection](https://colab.research.google.com/drive/1OdhhV057bx5SmkKs4LG1UKrTQtSZ4bXc#scrollTo=aca529ba):** Testing the limits of decision-making and metabolic costs.

In [ ]:
#@title F0: Initialize Environment (Realistic HH) { display-mode: "form" }
#@markdown This cell installs NEURON and defines the core 'Engine' functions.

import os
import sys

# 1. Install NEURON if not present
try:
    import neuron
    from neuron import h, gui
    print("NEURON is already installed.")
except ImportError:
    print("NEURON not found. Installing...")
    os.system('pip install neuron')
    from neuron import h, gui
    print("NEURON installation complete.")

import matplotlib.pyplot as plt
import numpy as np

# 2. Updated Builder Function: Handles multiple cell types
def build_population(n, cell_type='exc', g_na=0.12, g_k=0.036):
    """
    Creates a list of n Hodgkin-Huxley sections.
    cell_type: 'exc' for Excitatory (Principal) or 'inh' for Inhibitory.
    """
    pop = []
    for i in range(n):
        sec = h.Section(name=f'{cell_type}_{i}')
        sec.L = 20
        sec.diam = 20

        # INSERT MECHANISMS
        sec.insert('hh')   # Active channels (na, k, gl)
        sec.insert('pas')  # Passive leak mechanism

        # Apply custom channel conductances if present
        if h.ismembrane('hh', sec=sec):
            sec.gnabar_hh = g_na
            sec.gkbar_hh = g_k

        # Differentiation: Inhibitory cells often have faster/leakier membranes
        if cell_type == 'inh':
            sec.g_pas = 0.0002  # Leak conductance
            sec.e_pas = -65     # Leak reversal
        else:
            sec.g_pas = 0.0001
            sec.e_pas = -65

        pop.append(sec)
    return pop


# 3. Helper: Tonic Current Clamp
def tonic_iclamp(section, amp=0.1, delay=5.0, duration=1e9):
    stim = h.IClamp(section(0.5))
    stim.delay = delay
    stim.dur = duration
    stim.amp = amp
    return stim

# 4. Helper: Recording Setup
def mk_rec(section):
    v = h.Vector().record(section(0.5)._ref_v)
    return v

# 5. Helper: Plotting Logic
def plot_traces(t, voltages, labels, title="Neural Activity"):
    plt.figure(figsize=(10, 5))
    for v, l in zip(voltages, labels):
        plt.plot(t, v, label=l)
    plt.xlabel('Time (ms)')
    plt.ylabel('Membrane Potential (mV)')
    plt.title(title)
    plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))
    plt.grid(True, alpha=0.3)
    plt.show()

print("F0 Engine initialized and ready.")

### **F1 — STM (excitatory all-to-all)**

**Idea**: A brief pulse to one excitatory unit enters a recurrent $E \to E$ network. Sufficient weight and excitatory $\tau$ produce post-stimulus persistence, a hallmark of Short-Term Memory (STM).

**What to change**  
*   **Weight ($E \to E$):** Approximately 0.004–0.010.
*   **$\tau$ (E synaptic decay):** 1–5 ms.
*   **Delay ($E \to E$):** 0.5–2 ms.
*   **Stimulus:** Pulse amplitude and duration in `brief_iclamp`.

**Sanity checks**  
1.  Observe if spiking continues after the external pulse ends.
2.  Report the **persistence duration** (time from pulse offset to the last recorded spike).

**Predict → verify**  
*   **Persistence:** Increasing weight or $\tau$ should lengthen persistence until it reaches a saturation regime.
*   **Stability:** Excessive values may yield synchronized or runaway spiking.

**Exercises**  
1.  Find a (weight, $\tau$) pair that yields **50–150 ms** of persistence; capture the raster/trace and record the duration.
2.  Double the membrane leak ($g_{pas}$) and re-measure; explain the change using RC integrator logic.
3.  Identify a parameter set that fails to sustain activity; state which specific lever is currently limiting the network.

---



In [ ]:
# @title F1: STM Challenge - Pulse Ignition { display-mode: "form" }
from ipywidgets import interactive_output, HBox, VBox, FloatSlider, Layout, Accordion, HTML, Checkbox
import matplotlib.pyplot as plt
import numpy as np
from neuron import h

def simulate_f1_toggle(w_ee, tau_e, delay, g_na, g_k, g_pas, stim_amp, network_mode):
    h('forall delete_section()')

    # Toggle between 1-cell (no loop) and 2-cell (ping-pong loop)
    N = 2 if network_mode else 1
    cells = build_population(N, g_na=g_na, g_k=g_k)

    v_rest = -65.0
    for c in cells:
        sections = c.all if hasattr(c, 'all') else [c]
        for sec in sections:
            if h.ismembrane('pas', sec=sec):
                sec.g_pas = g_pas
                sec.e_pas = v_rest
            if h.ismembrane('hh', sec=sec):
                sec.gnabar_hh = g_na
                sec.gkbar_hh = g_k

    # Only connect if we have 2 cells
    if network_mode and N > 1:
        syns, ncs = connect_all_to_all(cells, tau=tau_e, weight=w_ee, delay=delay)

    # Ignition Pulse
    stim_in = brief_iclamp(cells[0], delay=20, dur=20, amp=stim_amp)

    t = h.Vector().record(h._ref_t)
    recs = [mk_rec(c) for c in cells]

    def custom_init():
        h.v_init = v_rest
        for sec in h.allsec():
            sec.v = v_rest
            if h.ismembrane('hh', sec=sec):
                sec.m_hh = 0.0529; sec.h_hh = 0.5961; sec.n_hh = 0.3177

    finit = h.FInitializeHandler(custom_init)
    h.finitialize(v_rest)
    h.continuerun(250)

    plt.figure(figsize=(10, 4.8))
    plt.plot(t, np.array(recs[0]), label='Cell 0 (Input Recipient)', color='#1f77b4', lw=2)
    if network_mode:
        plt.plot(t, np.array(recs[1]) - 2, label='Cell 1 (Loop Partner)', color='#ff7f0e', lw=2, alpha=0.7)

    plt.title("F1: STM Challenge | " + ("2-Cell Recurrent Loop" if network_mode else "Single Cell (No Loop)"))
    plt.axvspan(20, 40, color='green', alpha=0.1, label='WRITE (Pulse)')
    plt.axhline(-20, color='red', linestyle='--', alpha=0.2, label='Threshold')

    plt.xlim(0, 250)
    plt.ylim(-85, 45); plt.ylabel('V (mV)'); plt.xlabel('Time (ms)')
    plt.legend(loc='upper right', fontsize='small'); plt.grid(alpha=0.2); plt.show()

# --- UI SETUP ---
key_content = HTML("""
<b>Variable Key:</b>
<ul>
  <li><b>Weight_EE:</b> Strength of the connection between the two cells.</li>
  <li><b>Network Mode:</b> When OFF, you see a single cell. When ON, the loop is closed.</li>
</ul>
""")
acc = Accordion(children=[key_content])
acc.set_title(0, 'Roadmap & Variable Key')
acc.selected_index = None

s_lay = Layout(width='140px', height='200px')
sa_s  = FloatSlider(value=0.35, min=0.0, max=0.8, step=0.01, description='Stim_Amp', orientation='vertical', layout=s_lay)
w_s   = FloatSlider(value=0.34, min=0.0, max=1.5, step=0.01, description='Weight_EE', orientation='vertical', layout=s_lay)
na_s  = FloatSlider(value=0.10, min=0.0, max=0.4, step=0.01, description='g_Na', orientation='vertical', layout=s_lay)
k_s   = FloatSlider(value=0.06, min=0.0, max=0.2, step=0.01, description='g_K', orientation='vertical', layout=s_lay)
pas_s = FloatSlider(value=0.0001, min=0.0, max=0.001, step=0.00001, description='g_pas', orientation='vertical', layout=s_lay, readout_format='.5f')
t_s   = FloatSlider(value=5.0, min=1.0, max=50.0, step=1.0, description='Tau_Syn', orientation='vertical', layout=s_lay)
d_s   = FloatSlider(value=19.1, min=0.1, max=30.0, step=0.1, description='Delay', orientation='vertical', layout=s_lay)

# Restore the 1-vs-2 cell toggle
net_check = Checkbox(value=False, description='Enable Network (2-Cell)')

params = {'w_ee': w_s, 'tau_e': t_s, 'delay': d_s, 'g_na': na_s,
          'g_k': k_s, 'g_pas': pas_s, 'stim_amp': sa_s, 'network_mode': net_check}

display(VBox([acc, net_check, HBox([sa_s, w_s, na_s, k_s, pas_s, t_s, d_s]),
              interactive_output(simulate_f1_toggle, params)]))

# **W — Winner-Take-All (WTA) via Mutual Inhibition**

### **1. The Biological Concept**
In this module, we move from single-cell persistence to a network-level decision: **Reciprocal (Mutual) Inhibition**. This is a circuit motif found throughout the brain, from the spinal cord to the cortex.

In this architecture, each excitatory Principal cell has a "dedicated" inhibitory partner. When a Principal cell fires, it recruits its partner to silence all *other* competitors. This creates a "Hard WTA" state where the network must commit to a single "winner" based on the strongest input.

### **2. The "Levers" (Network Parameters)**
| Variable | Role in this Module | Unit | Instructional Role |
| :--- | :--- | :--- | :--- |
| **`E2I_weight`** | E → I coupling | μS | How easily a cell recruits its "defensive" interneuron. |
| **`II_weight`** | I → E competition | μS | The strength of the "veto." High values lead to faster, crisper decisions. |
| **`Input Gradient`** | $\Delta$ Excitation | nA | The "unfairness" of the race. Represents signal contrast. |

### **3. Predict → Verify**
> **Scenario:** We are providing a linear gradient of current from **Cell 0 (0.08 nA)** to **Cell 4 (0.12 nA)**.
> * **Predict:** If you set the `II_weight` to `0`, what will the raster plot look like?
> * **Verify:** Run the simulation below. Does the cell with the highest excitation successfully silence the others? How long does the "race" last before a winner is declared?

In [1]:
#@title W2: Mutual (I-I) Inhibition Architecture { display-mode: "form" }
#@markdown ### Network Configuration
N = 5
II_weight = 0.18 #@param {type:"slider", min:0, max:0.5, step:0.01}
E2I_weight = 0.01 #@param {type:"slider", min:0, max:0.05, step:0.001}
syn_delay = 0.5 #@param {type:"slider", min:0.1, max:5.0, step:0.1}

#@markdown ### Individual Excitation (Input Amps)
#@markdown Adjust these to decide which cell "wins" the race.
amp_0 = 0.08 #@param {type:"slider", min:0, max:0.2, step:0.01}
amp_1 = 0.18 #@param {type:"slider", min:0, max:0.2, step:0.01}
amp_2 = 0.03 #@param {type:"slider", min:0, max:0.2, step:0.01}
amp_3 = 0.01 #@param {type:"slider", min:0, max:0.2, step:0.01}
amp_4 = 0.01 #@param {type:"slider", min:0, max:0.2, step:0.01}

#@markdown ### Cell Biophysics
g_na = 0.12 #@param {type:"slider", min:0, max:0.5, step:0.01}
g_k = 0.036 #@param {type:"slider", min:0, max:0.1, step:0.005}

import numpy as np
from neuron import h
import matplotlib.pyplot as plt

# 1. BUILD
princ = build_population(N, cell_type='exc')
inh_cells = build_population(N, cell_type='inh')

for p in princ + inh_cells:
    p.gnabar_hh = g_na
    p.gkbar_hh = g_k

# 2. INPUTS: Using the individual sliders
amps = [amp_0, amp_1, amp_2, amp_3, amp_4]
ics = [tonic_iclamp(princ[i], amp=float(amps[i]), delay=5.0) for i in range(N)]

# 3. E -> I Trigger
for i in range(N):
    syn = h.ExpSyn(inh_cells[i](0.5))
    nc = h.NetCon(princ[i](0.5)._ref_v, syn, sec=princ[i])
    nc.weight[0] = E2I_weight
    nc.delay = 0.5

# 4. I -> E Competition (The Veto)
for i in range(N):
    for j in range(N):
        if i != j:
            syn = h.ExpSyn(princ[j](0.5))
            syn.e = -80.0
            nc = h.NetCon(inh_cells[i](0.5)._ref_v, syn, sec=inh_cells[i])
            nc.weight[0] = II_weight
            nc.delay = syn_delay

# --- RUN ---
t = h.Vector().record(h._ref_t)
recP = [mk_rec(p) for p in princ]
h.finitialize(-65)
h.continuerun(500)

# --- PLOT ---
plot_traces(np.array(t), [np.array(r) for r in recP], [f'P[{i}]' for i in range(N)],
            title=f'W2: WTA (II_weight={II_weight})')

Traceback (most recent call last):
  File "_pydevd_bundle/pydevd_cython.pyx", line 1078, in _pydevd_bundle.pydevd_cython.PyDBFrame.trace_dispatch
  File "_pydevd_bundle/pydevd_cython.pyx", line 297, in _pydevd_bundle.pydevd_cython.PyDBFrame.do_wait_suspend
  File "c:\Users\nairs\Anaconda3\envs\bmtk\lib\site-packages\debugpy\_vendored\pydevd\pydevd.py", line 1976, in do_wait_suspend
    keep_suspended = self._do_wait_suspend(thread, frame, event, arg, suspend_type, from_this_thread, frames_tracker)
  File "c:\Users\nairs\Anaconda3\envs\bmtk\lib\site-packages\debugpy\_vendored\pydevd\pydevd.py", line 2011, in _do_wait_suspend
    time.sleep(0.01)
KeyboardInterrupt


KeyboardInterrupt: 

### **Model Validation: Functional Connectivity Test**
Before analyzing the full 5-cell network, we validate the underlying synaptic logic. This test isolates two neurons to confirm that the inhibitory "veto" is functioning correctly.

* **The Goal:** Confirm that activity in **P[1]** (Trigger) successfully suppresses **P[0]** (Target).
* **Success Criteria:** A successful test is indicated by the target cell's membrane potential dropping below its resting level while the trigger cell is spiking.

In [ ]:
#@title Model Validation: Functional Connectivity Test { display-mode: "form" }
#@markdown This cell runs a "mini-simulation" to prove that P[1] can successfully silence P[0].

import numpy as np
from neuron import h
import matplotlib.pyplot as plt

# 1. Setup a minimal test (2 cells)
test_princ = build_population(2, cell_type='exc')
test_inh = build_population(2, cell_type='inh')

# 2. Wire the test circuit: P[1] -> I[1] -> P[0]
syn_ei = h.ExpSyn(test_inh[1](0.5))
nc_ei = h.NetCon(test_princ[1](0.5)._ref_v, syn_ei, sec=test_princ[1])
nc_ei.weight[0] = 0.01

syn_ie = h.ExpSyn(test_princ[0](0.5))
syn_ie.e = -80.0
nc_ie = h.NetCon(test_inh[1](0.5)._ref_v, syn_ie, sec=test_inh[1])
nc_ie.weight[0] = 0.18
nc_ie.delay = 1.0

# 3. Stimulate ONLY P[1]
stim = h.IClamp(test_princ[1](0.5))
stim.delay = 10
stim.dur = 50
stim.amp = 0.2

# Record
t = h.Vector().record(h._ref_t)
v0 = h.Vector().record(test_princ[0](0.5)._ref_v)
v1 = h.Vector().record(test_princ[1](0.5)._ref_v)

h.finitialize(-65)
h.continuerun(100)

# Plot Validation
plt.figure(figsize=(8, 4))
plt.plot(t, v1, label='Trigger Cell (P[1])', color='orange')
plt.plot(t, v0, label='Target Cell (P[0])', color='blue')
plt.title("Connectivity Validation: Reciprocal Inhibition Check")
plt.xlabel("Time (ms)")
plt.ylabel("Voltage (mV)")
plt.legend()
plt.show()

print(f"Validation Complete: If the Blue line (P[0]) dropped during the Orange spikes, the I-I Veto is active.")

## **Final Synthesis: The Cognitive LEGOs**

We have now explored two fundamental circuit motifs:

1.  **Short-Term Memory (F1):** Driven by **Recurrent Excitation (E→E)**. It allows a transient signal to persist, acting as a "buffer" for information.
2.  **Winner-Take-All (W):** Driven by **Mutual Inhibition (I→I)**. It allows a network to pick the strongest signal and suppress noise, acting as a "decision-maker."

**Discussion Question for Class:**
*In a real brain, these two motifs are often combined. If you had a network that had BOTH E→E persistence and I→I competition, how would that change the way you make a decision compared to a network that only has competition?*

## **Practice & Reflection: Small Networks (Set F)**

### **Challenge 1: Tuning the "Veto" Power**
In your **W — Winner-Take-All** cell, set the `II_weight` to **0.01** (very weak).
* **Observe:** Do you see "Secondary Winners" (multiple cells firing)?
* **Concept:** This is known as **Soft-WTA**. Why might a biological brain prefer a "Soft" decision (allowing multiple options) over a "Hard" one in certain scenarios?

### **Challenge 2: The Speed of Choice**
Reset your `II_weight` to **0.18**, but increase the `syn_delay` to **5.0 ms**.
* **Observe:** Look at the first 50ms of the simulation. Do the "loser" cells get more spikes out before being silenced?
* **Concept:** How does synaptic delay affect the **Reaction Time** and the "cleanliness" of a neural decision?

### **Challenge 3: The "Equal Input" Paradox**
Set all five `amp` sliders to exactly **0.10**.
* **Predict:** In a perfectly symmetrical model, who wins the race?
* **Verify:** Run the simulation. In a real biological brain, what "noise" factors might break this tie?

---

### **Discussion Questions**
1. **The Role of Leak:** In your F1 exploration, you saw how activity persists. If you doubled the membrane leak ($g_{pas}$), would it be easier or harder to sustain that activity? Explain using RC integrator logic.
2. **Structural Robustness:** In our WTA model, each Principal cell has one dedicated Inhibitory partner. If one of those Inhibitory cells "died" (was removed from the code), how would that change the competition for the remaining cells?
3. **Metabolic Cost:** In a "Hard-WTA" state, the winner fires at a very high frequency. Given that the $Na^+/K^+$ pump requires ATP to restore gradients after each spike, why is a "Hard" decision more metabolically expensive than a "Soft" one?
4. **Thresholding:** Why is the `nc.threshold` value critical for ensuring that only "true" spikes trigger inhibition, rather than just sub-threshold oscillations or noise?
5. **From Motifs to Systems:** We have built an **E→E** motif (Persistence) and an **I→I** motif (Competition). Describe a real-world task (e.g., choosing between two different paths while walking) where your brain might need to use *both* mechanisms simultaneously.